# Etapa 1 — Modelo Base de Redes Neuronales

**Semana 1: Línea Base de Desempeño**

En la primera fase se establece una línea base de desempeño para el sistema de clasificación de granos de café.

## Objetivos
- ✅ Seleccionar y documentar el dataset
- ✅ Realizar un Análisis Exploratorio de Datos (EDA)
- ✅ Preparar los datos para entrenamiento
- ✅ Construir un modelo neuronal base (MLP)
- ✅ Evaluar con métricas estándar

## 1. Configuración e Importaciones

Cargamos la configuración centralizada, inicializamos la semilla aleatoria y importamos los módulos necesarios del proyecto.

In [ ]:
import json
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from configs.config import get_config
from src.utils.seed import set_seed

# Cargar configuración del proyecto
cfg = get_config()
cfg.ensure_directories()
set_seed(cfg.seed)

print(f"🔧 Configuración cargada")
print(f"   - Dispositivo: {cfg.device}")
print(f"   - Semilla: {cfg.seed}")
print(f"   - Clases: {cfg.data.class_names}")
print(f"   - Tamaño de imagen: {cfg.data.image_size}×{cfg.data.image_size}")

## 2. Selección y Documentación del Dataset

### Fuente del Dataset
- **Nombre**: Dataset de Granos de Café
- **Ubicación**: `data/raw/coffee_beans/`
- **Estructura**: 
  - Train: `train/` (dividido por clases)
  - Test: `test/` (dividido por clases)

### Variables del Dataset
- **Entrada (X)**: Imágenes RGB de 128×128 píxeles
- **Salida (y)**: Categoría del grano (4 clases)
  - `dark`: Grano oscuro
  - `green`: Grano verde
  - `light`: Grano claro
  - `medium`: Grano medio

### Tamaño del Dataset

In [ ]:
from src.data.dataset import collect_samples

# Recolectar muestras de los conjuntos de entrenamiento y prueba
train_samples = collect_samples(cfg.paths.train_dir, cfg.data.class_names)
test_samples = collect_samples(cfg.paths.test_dir, cfg.data.class_names)

print(f"📊 Documentación del Dataset")
print(f"   - Muestras de entrenamiento: {len(train_samples)}")
print(f"   - Muestras de prueba: {len(test_samples)}")
print(f"   - Total de muestras: {len(train_samples) + len(test_samples)}")
print(f"   - Número de clases: {len(cfg.data.class_names)}")
print(f"   - Clases: {', '.join(cfg.data.class_names)}")

## 3. Análisis Exploratorio de Datos (EDA)

Analizamos la distribución de clases, visualizamos muestras, y calculamos estadísticas de color.

### 3.1 Distribución de Clases

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

# Función para calcular distribución de clases
def class_distribution(samples, class_names):
    """Retorna un dataframe con conteo de clases."""
    counts = Counter(sample.label for sample in samples)
    rows = [{"clase": class_name, "cantidad": counts[idx]} 
            for idx, class_name in enumerate(class_names)]
    return pd.DataFrame(rows)

# Calcular distribuciones
train_dist = class_distribution(train_samples, cfg.data.class_names)
test_dist = class_distribution(test_samples, cfg.data.class_names)

print("📈 Distribución de Clases en Entrenamiento:")
print(train_dist.to_string(index=False))
print("\n📈 Distribución de Clases en Prueba:")
print(test_dist.to_string(index=False))

### 3.2 Visualización de Muestras de Imágenes

In [ ]:
from PIL import Image

# Función para visualizar muestras de imágenes
def plot_sample_images(samples, class_names, max_images=8):
    """Visualiza una grilla de imágenes de muestra."""
    selected = list(samples[:max_images])
    columns = min(4, len(selected))
    rows = int(np.ceil(len(selected) / columns)) if columns else 1
    
    fig, axes = plt.subplots(rows, columns, figsize=(3 * columns, 3 * rows))
    axes = np.array(axes).reshape(-1)
    
    for idx, axis in enumerate(axes):
        if idx >= len(selected):
            axis.axis("off")
            continue
        record = selected[idx]
        image = Image.open(record.image_path).convert("RGB")
        axis.imshow(image)
        axis.set_title(f"Clase: {class_names[record.label]}")
        axis.axis("off")
    
    plt.tight_layout()
    return fig

# Mostrar muestras de entrenamiento
fig = plot_sample_images(train_samples, cfg.data.class_names)
plt.suptitle("Muestras de Imágenes del Dataset", fontsize=14, y=0.98)
plt.show()

### 3.3 Estadísticas de Color (RGB)

In [ ]:
# Función para calcular estadísticas RGB
def compute_rgb_statistics(samples, max_images=200):
    """Calcula media y desviación estándar por canal RGB."""
    subset = list(samples[:max_images])
    channels = []
    
    for sample in subset:
        image = np.array(Image.open(sample.image_path).convert("RGB"), dtype=np.float32) / 255.0
        channels.append(image.reshape(-1, 3))
    
    if not channels:
        return {"media_r": 0.0, "media_g": 0.0, "media_b": 0.0, 
                "std_r": 0.0, "std_g": 0.0, "std_b": 0.0}
    
    stacked = np.concatenate(channels, axis=0)
    mean = stacked.mean(axis=0)
    std = stacked.std(axis=0)
    
    return {
        "media_r": float(mean[0]),
        "media_g": float(mean[1]),
        "media_b": float(mean[2]),
        "std_r": float(std[0]),
        "std_g": float(std[1]),
        "std_b": float(std[2]),
    }

# Calcular estadísticas
rgb_stats = compute_rgb_statistics(train_samples)

print("🎨 Estadísticas de Color (RGB) del Dataset de Entrenamiento")
print(f"   Media R: {rgb_stats['media_r']:.4f} | Desv. Est. R: {rgb_stats['std_r']:.4f}")
print(f"   Media G: {rgb_stats['media_g']:.4f} | Desv. Est. G: {rgb_stats['std_g']:.4f}")
print(f"   Media B: {rgb_stats['media_b']:.4f} | Desv. Est. B: {rgb_stats['std_b']:.4f}")

### 3.4 Histogramas de Intensidad RGB

In [ ]:
# Función para visualizar histogramas RGB
def plot_color_histograms(samples, max_images=100):
    """Visualiza histogramas de intensidad RGB."""
    subset = list(samples[:max_images])
    
    if not subset:
        return None
    
    red_vals = []
    green_vals = []
    blue_vals = []
    
    for sample in subset:
        image = np.array(Image.open(sample.image_path).convert("RGB"))
        red_vals.append(image[:, :, 0].ravel())
        green_vals.append(image[:, :, 1].ravel())
        blue_vals.append(image[:, :, 2].ravel())
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(np.concatenate(red_vals), bins=50, color="red", alpha=0.4, label="Rojo")
    ax.hist(np.concatenate(green_vals), bins=50, color="green", alpha=0.4, label="Verde")
    ax.hist(np.concatenate(blue_vals), bins=50, color="blue", alpha=0.4, label="Azul")
    ax.set_title("Histogramas de Intensidad RGB")
    ax.set_xlabel("Intensidad de Píxel")
    ax.set_ylabel("Frecuencia")
    ax.legend()
    plt.tight_layout()
    return fig

# Mostrar histogramas
fig = plot_color_histograms(train_samples)
plt.show()

## 4. Preparación de los Datos

Configuramos transformaciones de imagen y creamos los DataLoaders para entrenamiento, validación y prueba.

### 4.1 Transformaciones de Imagen

In [ ]:
from src.data.transforms import get_train_transforms, get_eval_transforms

# Obtener transformaciones
train_transforms = get_train_transforms(image_size=cfg.data.image_size)
eval_transforms = get_eval_transforms(image_size=cfg.data.image_size)

print("✨ Transformaciones de Imagen Configuradas:")
print(f"   - Tamaño target: {cfg.data.image_size}×{cfg.data.image_size}")
print(f"   - Transformaciones de entrenamiento: Redimensionamiento, normalización")
print(f"   - Transformaciones de evaluación: Redimensionamiento, normalización")

### 4.2 División de Datos y DataLoaders

In [ ]:
from src.data.dataset import CoffeeBeansDataset, split_train_val_samples, get_dataloaders
from torch.utils.data import DataLoader

# Crear DataLoaders
dataloaders = get_dataloaders(cfg)

print("📦 DataLoaders Creados")
print(f"   - Tamaño de batch: {cfg.training.batch_size}")
print(f"   - Muestras de entrenamiento: {len(dataloaders['train'].dataset)}")
print(f"   - Muestras de validación: {len(dataloaders['val'].dataset)}")
print(f"   - Muestras de prueba: {len(dataloaders['test'].dataset)}")
print(f"   - Batches de entrenamiento: {len(dataloaders['train'])}")
print(f"   - Batches de validación: {len(dataloaders['val'])}")
print(f"   - Batches de prueba: {len(dataloaders['test'])}")

## 5. Construcción del Modelo MLP

Implementamos un Perceptrón Multicapa (MLP) como modelo base para clasificación.

In [ ]:
import torch.nn as nn

# Clase del Modelo MLP
class CoffeeMLP(nn.Module):
    """Perceptrón Multicapa para clasificación de granos de café."""

    def __init__(
        self,
        image_size: int,
        num_classes: int,
        hidden_dims: tuple = (512, 256),
        dropout: float = 0.3,
    ) -> None:
        super().__init__()
        
        # Calcular dimensión de entrada (3 canales × image_size × image_size)
        input_dim = 3 * image_size * image_size
        
        # Construir capas del modelo
        layers = [nn.Flatten()]
        previous_dim = input_dim
        
        # Capas ocultas con BatchNorm y Dropout
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(previous_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(inplace=True),
                nn.Dropout(p=dropout),
            ])
            previous_dim = hidden_dim
        
        # Capa de salida
        layers.append(nn.Linear(previous_dim, num_classes))
        
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

    def count_parameters(self) -> int:
        """Retorna el número de parámetros entrenables."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# Crear instancia del modelo
model = CoffeeMLP(
    image_size=cfg.data.image_size,
    num_classes=len(cfg.data.class_names),
    hidden_dims=cfg.model.hidden_dims,
    dropout=cfg.model.dropout,
)

print(f"🧠 Modelo MLP Creado")
print(f"   - Arquitectura: {cfg.model.hidden_dims}")
print(f"   - Dropout: {cfg.model.dropout}")
print(f"   - Parámetros entrenables: {model.count_parameters():,}")
print(f"\n{model}")

## 6. Entrenamiento del Modelo

Entrenamos el modelo con validación, early stopping y ajuste dinámico de tasa de aprendizaje.

### 6.1 Clase Entrenador

In [ ]:
from torch.optim import Optimizer
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader
from tqdm import tqdm
from dataclasses import dataclass

@dataclass
class SalidaEntrenamiento:
    """Contenedor para salidas del entrenamiento."""
    historia: dict
    ruta_mejor_checkpoint: Path

class Entrenador:
    """Entrenador del modelo con early stopping, scheduler y checkpointing."""

    def __init__(
        self,
        model: nn.Module,
        optimizer: Optimizer,
        criterion: nn.Module,
        device: str,
        ruta_checkpoint: Path,
        ruta_historia: Path,
        early_stopping_paciencia: int = 7,
        factor_scheduler: float = 0.5,
        paciencia_scheduler: int = 3,
    ) -> None:
        self.model = model.to(device)
        self.optimizer = optimizer
        self.criterion = criterion
        self.device = device
        self.ruta_checkpoint = ruta_checkpoint
        self.ruta_historia = ruta_historia
        self.early_stopping_paciencia = early_stopping_paciencia
        self.scheduler = ReduceLROnPlateau(
            optimizer=self.optimizer,
            mode="min",
            factor=factor_scheduler,
            patience=paciencia_scheduler,
        )

    def _ejecutar_epoca(self, dataloader: DataLoader, entrenamiento: bool) -> tuple:
        modo = "entrenamiento" if entrenamiento else "validación"
        self.model.train(entrenamiento)

        perdida_total = 0.0
        aciertos_totales = 0
        ejemplos_totales = 0

        for inputs, labels in tqdm(dataloader, desc=f"época ({modo})\", leave=False):
            inputs = inputs.to(self.device)
            labels = labels.to(self.device)

            if entrenamiento:
                self.optimizer.zero_grad()

            with torch.set_grad_enabled(entrenamiento):
                logits = self.model(inputs)
                perdida = self.criterion(logits, labels)
                if entrenamiento:
                    perdida.backward()
                    self.optimizer.step()

            tamaño_batch = labels.size(0)
            predicciones = torch.argmax(logits, dim=1)
            aciertos_totales += (predicciones == labels).sum().item()
            ejemplos_totales += tamaño_batch
            perdida_total += perdida.item() * tamaño_batch

        perdida_promedio = perdida_total / max(ejemplos_totales, 1)
        precision_promedio = aciertos_totales / max(ejemplos_totales, 1)
        return perdida_promedio, precision_promedio

    def entrenar(
        self,
        cargador_entrenamiento: DataLoader,
        cargador_validacion: DataLoader,
        epocas: int,
    ) -> SalidaEntrenamiento:
        """Entrena el modelo durante un número especificado de épocas."""
        self.ruta_checkpoint.parent.mkdir(parents=True, exist_ok=True)
        self.ruta_historia.parent.mkdir(parents=True, exist_ok=True)

        historia = {
            "perdida_entrenamiento": [],
            "precision_entrenamiento": [],
            "perdida_validacion": [],
            "precision_validacion": [],
        }
        mejor_perdida_validacion = float("inf")
        epocas_sin_mejora = 0

        for epoca in range(1, epocas + 1):
            # Ejecutar época de entrenamiento
            perdida_entrenamiento, precision_entrenamiento = self._ejecutar_epoca(
                cargador_entrenamiento, entrenamiento=True
            )
            # Ejecutar época de validación
            perdida_validacion, precision_validacion = self._ejecutar_epoca(
                cargador_validacion, entrenamiento=False
            )

            # Ajustar tasa de aprendizaje
            self.scheduler.step(perdida_validacion)

            # Guardar métricas
            historia["perdida_entrenamiento"].append(perdida_entrenamiento)
            historia["precision_entrenamiento"].append(precision_entrenamiento)
            historia["perdida_validacion"].append(perdida_validacion)
            historia["precision_validacion"].append(precision_validacion)

            print(
                f"Época {epoca:02d}/{epocas} | "
                f"pérd_entreno={perdida_entrenamiento:.4f} prec_entreno={precision_entrenamiento:.4f} | "
                f"pérd_valid={perdida_validacion:.4f} prec_valid={precision_validacion:.4f}"
            )

            # Early stopping y checkpointing
            if perdida_validacion < mejor_perdida_validacion:
                mejor_perdida_validacion = perdida_validacion
                epocas_sin_mejora = 0
                torch.save(self.model.state_dict(), self.ruta_checkpoint)
                print(f"   ✅ Nuevo mejor checkpoint guardado")
            else:
                epocas_sin_mejora += 1

            if epocas_sin_mejora >= self.early_stopping_paciencia:
                print(f"⏹️  Early stopping activado después de {epocas_sin_mejora} épocas sin mejora.")
                break

        # Guardar historia
        with self.ruta_historia.open("w", encoding="utf-8") as archivo:
            json.dump(historia, archivo, indent=2)

        return SalidaEntrenamiento(historia=historia, ruta_mejor_checkpoint=self.ruta_checkpoint)

print("✅ Clase Entrenador definida")

### 6.2 Configurar Optimizador y Entrenador

In [ ]:
from torch.optim import Adam

# Configurar optimizador
optimizer = Adam(
    model.parameters(),
    lr=cfg.training.learning_rate,
    weight_decay=cfg.training.weight_decay,
)

# Función de pérdida para clasificación
criterion = nn.CrossEntropyLoss()

# Crear instancia del entrenador
entrenador = Entrenador(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    device=cfg.device,
    ruta_checkpoint=cfg.paths.models_dir / "stage1_mlp_best.pt",
    ruta_historia=cfg.paths.results_dir / "stage1_training_history.json",
    early_stopping_paciencia=cfg.training.early_stopping_patience,
    factor_scheduler=cfg.training.scheduler_factor,
    paciencia_scheduler=cfg.training.scheduler_patience,
)

print("⚙️  Configuración de Entrenamiento")
print(f"   - Optimizador: Adam")
print(f"   - Tasa de aprendizaje: {cfg.training.learning_rate}")
print(f"   - Decay de peso (L2): {cfg.training.weight_decay}")
print(f"   - Función de pérdida: CrossEntropyLoss")
print(f"   - Épocas: {cfg.training.epochs}")
print(f"   - Paciencia Early Stopping: {cfg.training.early_stopping_patience}")

### 6.3 Ejecutar Entrenamiento

In [ ]:
# Establecer precisión de cálculo float32 (mejor rendimiento en GPU)
torch.set_float32_matmul_precision("high")

# Ejecutar entrenamiento
print("🚀 Iniciando entrenamiento...\n")
salida_entrenamiento = entrenador.entrenar(
    cargador_entrenamiento=dataloaders["train"],
    cargador_validacion=dataloaders["val"],
    epocas=cfg.training.epochs,
)

print(f"\n✅ Entrenamiento completado")
print(f"   - Mejor checkpoint guardado en: {salida_entrenamiento.ruta_mejor_checkpoint}")

### 6.4 Visualizar Curvas de Entrenamiento

In [ ]:
# Visualizar curvas de entrenamiento
historia = salida_entrenamiento.historia
epocas_rango = np.arange(1, len(historia["perdida_entrenamiento"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Gráfico de pérdida
axes[0].plot(epocas_rango, historia["perdida_entrenamiento"], label="Pérdida Entrenamiento", marker=".")
axes[0].plot(epocas_rango, historia["perdida_validacion"], label="Pérdida Validación", marker=".")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Pérdida (CrossEntropyLoss)")
axes[0].set_title("Curva de Pérdida")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Gráfico de precisión
axes[1].plot(epocas_rango, historia["precision_entrenamiento"], label="Precisión Entrenamiento", marker=".")
axes[1].plot(epocas_rango, historia["precision_validacion"], label="Precisión Validación", marker=".")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Precisión (Accuracy)")
axes[1].set_title("Curva de Precisión")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(cfg.paths.figures_dir / "stage1_training_curves.png", dpi=150)
print("📊 Curvas guardadas en:", cfg.paths.figures_dir / "stage1_training_curves.png")
plt.show()

## 7. Evaluación y Métricas

Evaluamos el modelo entrenado en el conjunto de prueba con análisis detallados.

### 7.1 Funciones de Métricas de Clasificación

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
import seaborn as sns

def calcular_metricas_clasificacion(
    y_verdadero,
    y_predicho,
    nombres_clases,
):
    """Calcula métricas estándar de clasificación."""
    reporte_dict = classification_report(
        y_verdadero,
        y_predicho,
        target_names=list(nombres_clases),
        output_dict=True,
        zero_division=0,
    )
    
    return {
        "exactitud": accuracy_score(y_verdadero, y_predicho),
        "precision_ponderada": precision_score(y_verdadero, y_predicho, average="weighted", zero_division=0),
        "precision_macro": precision_score(y_verdadero, y_predicho, average="macro", zero_division=0),
        "recall_ponderado": recall_score(y_verdadero, y_predicho, average="weighted", zero_division=0),
        "recall_macro": recall_score(y_verdadero, y_predicho, average="macro", zero_division=0),
        "f1_ponderado": f1_score(y_verdadero, y_predicho, average="weighted", zero_division=0),
        "f1_macro": f1_score(y_verdadero, y_predicho, average="macro", zero_division=0),
        "reporte_clasificacion": reporte_dict,
    }

print("✅ Funciones de métricas definidas")

### 7.2 Realizar Predicciones en Conjunto de Prueba

In [ ]:
# Cargar mejor checkpoint
model.load_state_dict(torch.load(cfg.paths.models_dir / "stage1_mlp_best.pt", map_location=cfg.device))
model.eval()

# Realizar predicciones en el conjunto de prueba
y_verdadero = []
y_predicho = []

with torch.no_grad():
    for inputs, labels in tqdm(dataloaders["test"], desc="Evaluando en conjunto de prueba"):
        inputs = inputs.to(cfg.device)
        labels = labels.to(cfg.device)
        
        outputs = model(inputs)
        predicciones = torch.argmax(outputs, dim=1)
        
        y_verdadero.extend(labels.cpu().numpy())
        y_predicho.extend(predicciones.cpu().numpy())

y_verdadero = np.array(y_verdadero)
y_predicho = np.array(y_predicho)

print("✅ Predicciones completadas")

### 7.3 Resultados de Métricas Principales

In [ ]:
# Calcular métricas
metricas = calcular_metricas_clasificacion(
    y_verdadero,
    y_predicho,
    cfg.data.class_names,
)

print("\n" + "="*60)
print("📊 MÉTRICAS DE EVALUACIÓN - CONJUNTO DE PRUEBA")
print("="*60)
print(f"\n🎯 Exactitud (Accuracy): {metricas['exactitud']:.4f} ({metricas['exactitud']*100:.2f}%)")
print(f"\n📈 Precision:")
print(f"   - Ponderada: {metricas['precision_ponderada']:.4f}")
print(f"   - Macro: {metricas['precision_macro']:.4f}")
print(f"\n🎪 Recall:")
print(f"   - Ponderada: {metricas['recall_ponderado']:.4f}")
print(f"   - Macro: {metricas['recall_macro']:.4f}")
print(f"\n📍 F1-Score:")
print(f"   - Ponderada: {metricas['f1_ponderado']:.4f}")
print(f"   - Macro: {metricas['f1_macro']:.4f}")
print("="*60)

### 7.4 Reporte Detallado por Clase

In [ ]:
# Mostrar reporte de clasificación detallado
reporte_dict = metricas["reporte_clasificacion"]

print("\n📋 REPORTE DETALLADO POR CLASE")
print("="*80)

for clase_idx, clase_nombre in enumerate(cfg.data.class_names):
    if clase_nombre in reporte_dict:
        metricas_clase = reporte_dict[clase_nombre]
        print(f"\n🫘 Clase: {clase_nombre.upper()}")
        print(f"   - Precision: {metricas_clase['precision']:.4f}")
        print(f"   - Recall: {metricas_clase['recall']:.4f}")
        print(f"   - F1-Score: {metricas_clase['f1-score']:.4f}")
        print(f"   - Muestras: {int(metricas_clase['support'])}")

print("="*80)

### 7.5 Matriz de Confusión

In [ ]:
# Calcular y visualizar matriz de confusión
matriz_confusion = confusion_matrix(y_verdadero, y_predicho)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    matriz_confusion,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=cfg.data.class_names,
    yticklabels=cfg.data.class_names,
    ax=ax,
)
ax.set_xlabel("Predicción", fontsize=12)
ax.set_ylabel("Verdadero", fontsize=12)
ax.set_title("Matriz de Confusión - Conjunto de Prueba", fontsize=14)
plt.tight_layout()
plt.savefig(cfg.paths.figures_dir / "stage1_confusion_matrix.png", dpi=150)
print("📊 Matriz de confusión guardada en:", cfg.paths.figures_dir / "stage1_confusion_matrix.png")
plt.show()

### 7.6 Guardar Resultados

In [ ]:
# Guardar resultados en JSON
resultados_finales = {
    "exactitud": float(metricas["exactitud"]),
    "precision_ponderada": float(metricas["precision_ponderada"]),
    "precision_macro": float(metricas["precision_macro"]),
    "recall_ponderado": float(metricas["recall_ponderado"]),
    "recall_macro": float(metricas["recall_macro"]),
    "f1_ponderado": float(metricas["f1_ponderado"]),
    "f1_macro": float(metricas["f1_macro"]),
    "reporte_clasificacion": metricas["reporte_clasificacion"],
    "configuracion": {
        "modelo": "CoffeeMLP",
        "capas_ocultas": cfg.model.hidden_dims,
        "dropout": cfg.model.dropout,
        "parametros_entrenables": model.count_parameters(),
        "optimizador": "Adam",
        "tasa_aprendizaje": cfg.training.learning_rate,
        "epocas_entrenadas": len(historia["perdida_entrenamiento"]),
    },
}

ruta_resultados = cfg.paths.results_dir / "stage1_test_metrics.json"
with open(ruta_resultados, "w", encoding="utf-8") as f:
    json.dump(resultados_finales, f, indent=2)

print(f"✅ Resultados guardados en: {ruta_resultados}")

## Resumen Final

### ✅ Actividades Completadas

1. **Documentación del Dataset**
   - Fuente: Dataset de Granos de Café
   - Tamaño: Múltiples imágenes RGB de 128×128 píxeles
   - Clases: 4 (dark, green, light, medium)

2. **Análisis Exploratorio de Datos (EDA)**
   - Distribución de clases en entrenamiento y prueba
   - Visualización de muestras de imágenes
   - Estadísticas RGB y histogramas de color

3. **Preparación de Datos**
   - Transformaciones de imagen (redimensionamiento, normalización)
   - División estratificada train/val/test
   - DataLoaders configurados

4. **Modelo Neuronal Base (MLP)**
   - Arquitectura: Entrada (3×128×128) → Capas ocultas (512, 256) → Salida (4 clases)
   - Técnicas: BatchNorm, ReLU, Dropout

5. **Entrenamiento**
   - Optimizador: Adam
   - Early stopping con paciencia
   - Ajuste dinámico de tasa de aprendizaje

### 📊 Métricas de Evaluación

**Métricas Principales:**
- **Exactitud (Accuracy)**: Proporción global de predicciones correctas
- **Precision**: Proporción de predicciones positivas que fueron correctas  
- **Recall**: Proporción de muestras positivas que fueron identificadas
- **F1-Score**: Media armónica de Precision y Recall

**Tipos de Cálculo:**
- **Ponderada**: Promedio ponderado por el número de muestras por clase
- **Macro**: Promedio simple de todas las clases (sin ponderación)